In [7]:
from pathlib import Path

from dotenv import load_dotenv

for env_path in (Path(".env"), Path("../.env"), Path("../../.env")):
    if env_path.exists():
        load_dotenv(env_path, override=True)
        break

In [8]:
from langchain_mcp_adapters.client import MultiServerMCPClient

client = MultiServerMCPClient(
    {
        "travel_server": {
            "transport": "streamable_http",
            "url": "https://mcp.kiwi.com",
        }
    }
)

tools = await client.get_tools()
print(f"Loaded {len(tools)} travel tools")
for tool in tools:
    print(f"- {tool.name}: {tool.description}")

Loaded 2 travel tools
- search-flight: # Search for flights

Searches Kiwi.com for available flights between two locations for the given dates and passengers. City or airport names are resolved automatically, so call this whenever the user wants to search for flights — whether they gave IATA codes or just place names.

## Result shape

Returns `{ query, currency, passengers, resultsCount, itineraries, searchTimeMs }`. Each item in `itineraries` has:
- `price` (number) and `priceFormatted` (e.g. "123 EUR")
- `totalDurationSeconds`
- `bookingUrl` — the link to book the flight
- `imageId` — destination city id for a hero photo (https://images.kiwi.com/photos/600x600/{imageId}.jpg)
- `baggage` — total included baggage across all travelers: `{ personalItem, cabinBag, checkedBag }` (counts)
- `outbound` (and `inbound` for return flights), each a leg with: `route` (list of airport codes including layovers, e.g. ["PRG","MAD","BCN"]), `departureTime` / `arrivalTime` (local ISO timestamps), `dur

In [9]:
from langchain.agents import create_agent
from langgraph.checkpoint.memory import InMemorySaver

agent = create_agent(
    model="groq:openai/gpt-oss-20b",
    tools=tools,
    checkpointer=InMemorySaver(),
    system_prompt="You are a travel agent. Do not ask follow up questions unless the user has not provided enough information to make a useful search.",
)

In [10]:
from langchain.messages import HumanMessage

config = {"configurable": {"thread_id": "travel-1"}}

response = await agent.ainvoke(
    {
        "messages": [
            HumanMessage(
                content="Get me a direct flight from San Francisco to Tokyo on March 31st."
            )
        ]
    },
    config=config,
)

In [11]:
from pprint import pprint

pprint(response)

{'messages': [HumanMessage(content='Get me a direct flight from San Francisco to Tokyo on March 31st.', additional_kwargs={}, response_metadata={}, id='6655e26e-3dad-45c4-b777-701bcae1f998'),
              AIMessage(content='', additional_kwargs={'reasoning_content': 'The user wants a direct flight from San Francisco to Tokyo on March 31. We need to search flight. Provide flight options. The user didn\'t specify return. So one-way. They didn\'t specify number of adults, but default 1 adult. Use default cabinClass? default is null. Use default currency EUR? Maybe better to use USD or local. But default is EUR. We can use default. Provide flight search. Use search-flight function.\n\nWe need to provide output in markdown table with details. Group into cheapest, shortest, others.\n\nWe need to call search-flight with parameters: flyFrom: "San Francisco" or "SFO". flyTo: "Tokyo" or "NRT" or "HND". Use "San Francisco" and "Tokyo". departureDate: "31/03/2024". departureDateFlexDays: 0. We ne

In [12]:
print(response["messages"][-1].content)

I’m sorry, but March 31, 2024 is already in the past. Could you let me know a future date (or a date range) you’d like to fly from San Francisco to Tokyo? Once I have that, I’ll find the best direct flights for you. Safe travels!
